# Bronze Layer Ingestion Notebook

This notebook ingests raw CSV files into the Bronze layer as Delta tables.

## Tools and Components Used
- **Databricks + PySpark**: read CSV files and run distributed ingestion
- **Delta Lake**: persist raw data as managed Delta tables
- **Unity Catalog** (`workspace.bronze`): register Bronze tables for downstream layers

## What This Notebook Does
1. Defines source file configuration for CRM and ERP datasets
2. Reads each CSV with schema inference
3. Writes raw data to Bronze Delta tables in overwrite mode
4. Runs a quick validation to confirm table creation


## 1) Configuration

Set file paths and target namespace once, then keep ingestion rules in one config list.


In [ ]:
BASE_PATH = "/Volumes/workspace/bronze/source_data"
CATALOG = "workspace"
SCHEMA = "bronze"
BRONZE_NAMESPACE = f"{CATALOG}.{SCHEMA}"

LOAD_CONFIG = [
    # CRM sources
    {"source": "crm", "path": f"{BASE_PATH}/crm/cust_info.csv", "table": "crm_cust_info_raw"},
    {"source": "crm", "path": f"{BASE_PATH}/crm/prd_info.csv", "table": "crm_prd_info_raw"},
    {"source": "crm", "path": f"{BASE_PATH}/crm/sales_details.csv", "table": "crm_sales_details_raw"},

    # ERP sources
    {"source": "erp", "path": f"{BASE_PATH}/erp/CUST_AZ12.csv", "table": "erp_cust_az12_raw"},
    {"source": "erp", "path": f"{BASE_PATH}/erp/LOC_A101.csv", "table": "erp_loc_a101_raw"},
    {"source": "erp", "path": f"{BASE_PATH}/erp/PX_CAT_G1V2.csv", "table": "erp_px_cat_g1v2_raw"},
]


## 2) Ingestion Logic

Wrap read/write steps in a function so the ingestion flow is reusable and easier to debug.


In [ ]:
def load_csv_to_bronze(config: dict) -> None:
    source = config["source"]
    path = config["path"]
    table = config["table"]
    full_table_name = f"{BRONZE_NAMESPACE}.{table}"

    print(f"[START] Loading {source.upper()} file: {path}")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

    (
        df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(full_table_name)
    )

    print(f"[DONE] Wrote table: {full_table_name}")


## 3) Execute Bronze Loads


In [ ]:
for config in LOAD_CONFIG:
    load_csv_to_bronze(config)


## 4) Validation

Quickly check that Bronze tables are available in the target schema.


In [ ]:
display(
    spark.sql(f"SHOW TABLES IN {BRONZE_NAMESPACE}")
    .select("tableName", "isTemporary")
    .orderBy("tableName")
)
